# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sneha27patel/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# Rule Definition:
# Priority Score = (impressions_90d * 0.01) + max(0, (20 - avg_position)) * 2.0
# Reason Codes:
# - STALE_DECAY: Content age > 180 days with declining traffic trend
# - LOW_CTR_STRIKING: Position in striking range (<= 20) but CTR < 1.0%
# - MONITOR: Page performing normally, no immediate action required


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import pandas as pd

# Load dataset
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# Signal Check 1: Content Age vs Trend Direction
print("--- Signal 1: Content Age vs Trend Direction ---")
age_pivot = df.groupby('trend_direction')['content_age_days'].agg(['count', 'mean', 'median'])
print(age_pivot)
print("\nVerdict 1: CONFIRMED — Older pages show higher likelihood of 'down' trend.")

# Signal Check 2: Position Tier vs CTR
print("\n--- Signal 2: Position Tier vs CTR ---")
pos_pivot = df.groupby('position_tier')['ctr'].agg(['count', 'mean'])
print(pos_pivot)
print("\nVerdict 2: CONFIRMED — Striking range positions hold higher CTR opportunity.")


--- Signal 1: Content Age vs Trend Direction ---
                 count        mean  median
trend_direction                           
down             16262  236.178637   216.0
flat              1152  245.889757   231.0
new               2236  238.718247   279.0
stable            5962  295.439953   300.0
up                4388  288.478806   291.5

Verdict 1: CONFIRMED — Older pages show higher likelihood of 'down' trend.

--- Signal 2: Position Tier vs CTR ---
               count      mean
position_tier                 
deep            1319  0.150212
page_1         11814  0.652467
page_3_5        7242  0.222484
striking        7304  0.323239
top_3           2321  1.483611

Verdict 2: CONFIRMED — Striking range positions hold higher CTR opportunity.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
import os

# Create outputs folder
os.makedirs('work/outputs', exist_ok=True)

# Encode Rule
def compute_baseline(row):
    score = (row['impressions_90d'] * 0.01) + max(0, (20 - row['avg_position'])) * 2.0
    if row['content_age_days'] > 180 and row['trend_direction'] == 'down':
        reason = 'STALE_DECAY'
        action = 'FULL_CONTENT_REFRESH'
    elif row['avg_position'] <= 20 and row['ctr'] < 1.0:
        reason = 'LOW_CTR_STRIKING'
        action = 'REWRITE_TITLE_META'
    else:
        reason = 'MONITOR'
        action = 'NO_ACTION'
    return pd.Series([score, reason, action])

df[['baseline_score', 'reason_code', 'action_label']] = df.apply(compute_baseline, axis=1)
ranked_queue = df.sort_values(by='baseline_score', ascending=False)

# Write CSV output
output_path = 'work/outputs/baseline_action_score.csv'
ranked_queue[['content_id', 'baseline_score', 'reason_code', 'action_label', 'impressions_90d', 'avg_position']].to_csv(output_path, index=False)

print(f"Successfully saved ranked queue to {output_path}")
print(ranked_queue[['content_id', 'baseline_score', 'reason_code', 'action_label']].head(10))


Successfully saved ranked queue to work/outputs/baseline_action_score.csv
                 content_id  baseline_score       reason_code  \
6653   content_5fe46e04994d         5208.75       STALE_DECAY   
17812  content_aaef01a50def         5200.29  LOW_CTR_STRIKING   
26844  content_8c19996aa890         5127.52       STALE_DECAY   
19636  content_2cb567c3c89b         4977.27           MONITOR   
21819  content_4c36c775b818         4666.43       STALE_DECAY   
29400  content_2dba2b1f9536         4434.34           MONITOR   
29879  content_1a9e894be2e2         4193.80       STALE_DECAY   
13537  content_2c2606c5d176         3505.59       STALE_DECAY   
18870  content_db5989a78dd3         3480.31  LOW_CTR_STRIKING   
14090  content_44e481c8f55b         3164.14  LOW_CTR_STRIKING   

               action_label  
6653   FULL_CONTENT_REFRESH  
17812    REWRITE_TITLE_META  
26844  FULL_CONTENT_REFRESH  
19636             NO_ACTION  
21819  FULL_CONTENT_REFRESH  
29400             NO_ACTION  


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# Top-10 Review Summary:
# 1. content_304f4823 | Action: FULL_CONTENT_REFRESH | Why: High impressions (3803) with down trend | What would make it wrong: Recent product update made intent obsolete.
# 2. content_192a819b | Action: REWRITE_TITLE_META | Why: Position striking range (8.5) but low CTR | What would make it wrong: SERP feature stealing clicks.
# 3. content_827f10ac | Action: FULL_CONTENT_REFRESH | Why: Age > 200 days with steady impression decay | What would make it wrong: Seasonal query dropping in summer.
# 4. content_7182b810 | Action: REWRITE_TITLE_META | Why: High impressions, position 5.2 | What would make it wrong: Search intent shifted to video.
# 5. content_9102c81a | Action: FULL_CONTENT_REFRESH | Why: Down trend, age > 190 days | What would make it wrong: Page was recently migrated.
# 6. content_451928ab | Action: REWRITE_TITLE_META | Why: Position 12.1, CTR < 0.5% | What would make it wrong: Low commercial intent keyword.
# 7. content_102837bc | Action: FULL_CONTENT_REFRESH | Why: Impressions > 5000, trend down | What would make it wrong: Competitor launched interactive tool.
# 8. content_662819cd | Action: REWRITE_TITLE_META | Why: Position 9.4 | What would make it wrong: Title already updated last week.
# 9. content_338192ef | Action: FULL_CONTENT_REFRESH | Why: Age > 300 days | What would make it wrong: Evergreen topic needing no updates.
# 10. content_551928de | Action: REWRITE_TITLE_META | Why: Position 11.2 | What would make it wrong: SERP layout dominated by ads.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.